## 거래처 계층 구조 테이블 생성
- 오타 정제 → 입고처 제거 → Parent/Child 분류 → Duplicate 통합 → ID 재부여

In [1]:
import re
import msoffcrypto
import io
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# 0. 파일 로드
# ─────────────────────────────────────────────
FILE_PATH = "통합170901DATA취합.xlsx"
PASSWORD  = "33"

with open(FILE_PATH, "rb") as f:
    office_file = msoffcrypto.OfficeFile(f)
    office_file.load_key(password=PASSWORD)
    decrypted = io.BytesIO()
    office_file.decrypt(decrypted)

wb = openpyxl.load_workbook(decrypted, read_only=True, data_only=True)
ws = wb["DATA"]
rows = list(ws.iter_rows(values_only=True))

columns = [
    "key", "영업담당자", "거래처명", "삼일모델명", "업체모델명", "CODE_No",
    "단가", "합의중량_g", "구단가_1",
    "구단가_2", "구단가_3", "구단가_4", "구단가_5", "구단가_6", "구단가_7",
    "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일", "제품군", "비고"
]

data = [row for row in rows[2:] if any(row)]
df_raw = pd.DataFrame(data).iloc[:, :31]
df_raw.columns = columns
df_raw = df_raw.drop(columns=["key"])
df_raw["거래처명"] = df_raw["거래처명"].astype(str).str.strip()
df_raw = df_raw[df_raw["거래처명"].notna() & (df_raw["거래처명"] != "None")].reset_index(drop=True)

print(f"원본 행 수: {len(df_raw)}")
print(f"원본 고유 거래처: {df_raw['거래처명'].nunique()}개")

원본 행 수: 5252
원본 고유 거래처: 900개


In [66]:
# ─────────────────────────────────────────────
# 1. 오타 정제
# 거래처명 전체(슬래시 앞/뒤 포함)에서 오타를 올바른 이름으로 교체
# 추가 오타 발견 시 여기에만 추가하면 됨
# ─────────────────────────────────────────────
TYPO_MAP = {
    "가지채움"      : "가치채움",
    "CJ대한퉁운"    : "CJ대한통운",
    "인펙코리아"    : "인팩코리아",
    "한국파렛트폴"   : "한국파렛트풀",
    "안팩코리아"    :  "인팩코리아",
    "중부스티로폴"  :  "중부스티로폼",
    "하이엠푸드(구서울우유치즈몰안양)" : "서울우유치즈몰안양",
    "천수푸드(절대육감)" : "천수푸드",
    "삼일이노팩&탭스인터내셔널" : "탭스인터내셔널",
    "한화테크원" : "한화테크윈",
    "코리아레지스토리" : "코리아레지스트리",
    "삘강맛푸드": "빨강맛푸드"
}

def normalize_name(name: str) -> str:
    """
    괄호 패턴 정규화
    - (유)/(주) 등 맨 앞 괄호  →  그대로
    - TYPO_MAP에 있는 것       →  먼저 교체 (구 상호명 등 예외 처리)
    - 회사명(지점/브랜드)      →  회사명/지점
    """
    name = name.strip()

    # 맨 앞 괄호 → 법인 형태 표기, 건드리지 않음
    if re.match(r'^\(', name):
        return name

    # TYPO_MAP 우선 적용
    if name in TYPO_MAP:
        return TYPO_MAP[name]

    # 회사명(브랜드/지점) → 회사명/브랜드
    match = re.match(r'^(.+?)\((.+?)\)$', name)
    if match:
        company = match.group(1).strip()
        branch  = match.group(2).strip()
        # 괄호 안이 "구"로 시작하면 → 회사명만 유지 (TYPO_MAP 미등록 케이스 대비)
        if branch.startswith("구"):
            return company
        return f"{company}/{branch}"

    return name

def fix_typo(name: str) -> str:
    """슬래시 앞/뒤 각각 정제 + 괄호 정규화"""
    if "/" in name:
        parts = name.split("/", 1)
        parts = [normalize_name(TYPO_MAP.get(p.strip(), p.strip())) for p in parts]
        return "/".join(parts)
    return normalize_name(name)

before = df_raw["거래처명"].copy()
df_raw["거래처명"] = df_raw["거래처명"].apply(fix_typo)

fixed = (before != df_raw["거래처명"]).sum()
print(f"오타 수정된 행: {fixed}개")
print(df_raw.loc[before != df_raw["거래처명"], "거래처명"].value_counts().to_string())

오타 수정된 행: 0개
Series([], )


In [451]:
# ─────────────────────────────────────────────
# 2. 입고처 처리
# 슬래시 앞 토큰이 입고처면 → 앞을 제거하고 뒤(진짜 거래처)만 남김
# ex) 용인공장/한국이노팩  →  한국이노팩
# ─────────────────────────────────────────────
INGGO_SET = {
    "강원공장",
    "둔포공장",
    "용인공장",
    "서해공장",
    "아산공장",
    "싱싱패키지",
}

def strip_inggo(name: str) -> str:
    """입고처가 앞에 있으면 제거하고 뒤 이름만 반환"""
    if "/" in name:
        parent_token = name.split("/")[0].strip()
        if parent_token in INGGO_SET:
            return name.split("/", 1)[1].strip()
    return name

before = df_raw["거래처명"].copy()
df_raw["거래처명"] = df_raw["거래처명"].apply(strip_inggo)

stripped = (before != df_raw["거래처명"]).sum()
print(f"입고처 제거된 행: {stripped}개")
# 변경 전후 확인
changed = before[before != df_raw["거래처명"]]
for old, new in zip(changed.values, df_raw.loc[changed.index, "거래처명"].values):
    print(f"  {old}  →  {new}")

입고처 제거된 행: 0개


In [4]:
# ─────────────────────────────────────────────
# 3. 메타 정보 추출 (정제 완료된 거래처명 기준)
# ─────────────────────────────────────────────
META_COLS = [
    "거래처명", "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일"
]
df_meta = df_raw[META_COLS].drop_duplicates(subset=["거래처명"]).set_index("거래처명")
print(f"정제 후 고유 거래처: {len(df_meta)}개")

정제 후 고유 거래처: 889개


In [5]:
# ─────────────────────────────────────────────
# 4. Master 테이블 구성
# ─────────────────────────────────────────────
all_clients = set(df_raw["거래처명"].unique())
slash_clients      = {c for c in all_clients if "/" in c}
standalone_clients = {c for c in all_clients if "/" not in c}
parent_names_from_slash = {c.split("/")[0].strip() for c in slash_clients}
auto_create_parents = parent_names_from_slash - standalone_clients

print(f"전체 고유 거래처  : {len(all_clients)}개")
print(f"  슬래시(/) 패턴  : {len(slash_clients)}개")
print(f"  단독 행         : {len(standalone_clients)}개")
print(f"  자동생성 Parent : {len(auto_create_parents)}개")

records = []

# Step A: 단독 행 (고객사)
for name in sorted(standalone_clients):
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명" : name,
        "parent_명": None,
        "node_type": "고객사",
        **{col: meta.get(col) for col in META_COLS[1:]}
    })

# Step B: 자동 생성 Parent (슬래시에만 등장)
for name in sorted(auto_create_parents):
    records.append({
        "거래처명" : name,
        "parent_명": None,
        "node_type": "고객사",
        **{col: None for col in META_COLS[1:]}
    })

# Step C: Child (납품처)
for name in sorted(slash_clients):
    parent_name = name.split("/")[0].strip()
    child_label = name.split("/", 1)[1].strip()
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명" : child_label,
        "parent_명": parent_name,
        "node_type": "납품처",
        **{col: meta.get(col) for col in META_COLS[1:]}
    })

df_master = pd.DataFrame(records).reset_index(drop=True)

# ID 부여
df_master.insert(0, "거래처_ID", range(1, len(df_master) + 1))
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

# parent_id 컬럼 위치 조정
cols = df_master.columns.tolist()
cols.remove("parent_id")
cols.insert(2, "parent_id")
df_master = df_master[cols]

# 정렬 (고객사 먼저, 납품처 뒤)
type_order = {"고객사": 0, "납품처": 1}
df_master["_sort"] = df_master["node_type"].map(type_order)
df_master = df_master.sort_values(["_sort", "거래처_ID"]).drop(columns=["_sort"]).reset_index(drop=True)
df_master["거래처_ID"] = range(1, len(df_master) + 1)
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

전체 고유 거래처  : 889개
  슬래시(/) 패턴  : 260개
  단독 행         : 629개
  자동생성 Parent : 32개


In [6]:
# ─────────────────────────────────────────────
# 5. Duplicate 통합
# 같은 거래처명이 고객사 + 납품처 둘 다 있는 경우:
#   → 납품처 행은 삭제하지 않고 유지
#   → 거래처_ID를 고객사 ID로 통일
#   → 사업자번호/주소 등 빈 칸은 고객사 데이터로 채우기
# ─────────────────────────────────────────────
고객사_names = set(df_master[df_master["node_type"] == "고객사"]["거래처명"])
납품처_names = set(df_master[df_master["node_type"] == "납품처"]["거래처명"])
duplicates = 고객사_names & 납품처_names

print(f"Duplicate 거래처: {len(duplicates)}개")
for name in sorted(duplicates):
    print(f"  • {name}")

for name in duplicates:
    real_row  = df_master[(df_master["거래처명"] == name) & (df_master["node_type"] == "고객사")].iloc[0]
    real_id   = real_row["거래처_ID"]
    mask_fake = (df_master["거래처명"] == name) & (df_master["node_type"] == "납품처")

    # 납품처 행의 거래처_ID를 고객사 ID로 통일하기 전에 기존 ID 저장
    old_ids = df_master.loc[mask_fake, "거래처_ID"].unique()

    # 거래처_ID를 고객사 ID로 통일
    df_master.loc[mask_fake, "거래처_ID"] = real_id

    # 기존 ID를 parent_id로 가리키던 다른 납품처들도 real_id로 재연결
    for old_id in old_ids:
        mask_children = df_master["parent_id"] == old_id
        df_master.loc[mask_children, "parent_id"] = real_id

    # 빈 칸만 고객사 정보로 채우기
    for col in ["사업자번호", "주소", "대표자", "업태", "종목"]:
        df_master.loc[mask_fake, col] = real_row[col]

print(f"\n정리 후 전체: {len(df_master)}개")
print(f"  고객사: {(df_master['node_type']=='고객사').sum()}개")
print(f"  납품처: {(df_master['node_type']=='납품처').sum()}개")

Duplicate 거래처: 18개
  • 건어맨
  • 남선푸드
  • 동아에스티
  • 디베랴
  • 로지스프로
  • 메이플델리
  • 명정보기술
  • 바이오플레이
  • 선우
  • 아이탐푸드
  • 우진수산
  • 유한산업
  • 정남
  • 제이월드텍
  • 지웅
  • 진원
  • 태진패키지
  • 팀프레시

정리 후 전체: 921개
  고객사: 661개
  납품처: 260개


In [7]:
# ─────────────────────────────────────────────
# 6. 청구대상_ID 추가
# 납품처 → default: parent_id (세금계산서는 parent에게 청구)
# 고객사 → 자기 자신 거래처_ID
# 나중에 예외 거래처는 이 컬럼만 수동 수정하면 됨
# ─────────────────────────────────────────────
df_master["청구대상_ID"] = df_master.apply(
    lambda row: row["parent_id"]
        if (row["node_type"] == "납품처" and pd.notna(row["parent_id"]))
        else row["거래처_ID"],
    axis=1
).astype("Int64")

# parent_id 바로 뒤에 위치
cols = df_master.columns.tolist()
cols.remove("청구대상_ID")
cols.insert(cols.index("parent_id") + 1, "청구대상_ID")
df_master = df_master[cols]

# 샘플 확인
print("【청구대상_ID 샘플 (납품처)】")
sample_cols = ["거래처_ID", "거래처명", "node_type", "parent_명", "parent_id", "청구대상_ID"]
print(df_master[df_master["node_type"] == "납품처"][sample_cols].head(10).to_string(index=False))

【청구대상_ID 샘플 (납품처)】
 거래처_ID     거래처명 node_type parent_명  parent_id  청구대상_ID
    662     보라티알       납품처   AJ네트웍스      630.0      630
    663   온누리스토어       납품처   AJ네트웍스      630.0      630
    664       정들       납품처   AJ네트웍스      630.0      630
    665       태광       납품처   AJ네트웍스      630.0      630
    666     하겐다즈       납품처   AJ네트웍스      630.0      630
    667    한강로직스       납품처   AJ네트웍스      630.0      630
    668     고려포장       납품처  ATEC AP      631.0      631
    669      가미락       납품처   CJ대한통운      632.0      632
    670 강원도옥수수총각       납품처   CJ대한통운      632.0      632
    671    그림엘푸드       납품처   CJ대한통운      632.0      632


In [8]:
# ─────────────────────────────────────────────
# 6. 검증
# ─────────────────────────────────────────────
print("【Parent → Child 샘플】")
for p_name in ["청호나이스", "CJ대한통운", "동원홈푸드", "한국파렛트풀", "한국이노팩", "컬리"]:
    if p_name not in name_to_id:
        print(f"  {p_name} → 없음")
        continue
    pid  = name_to_id[p_name]
    row  = df_master[df_master["거래처_ID"] == pid].iloc[0]
    kids = df_master[df_master["parent_id"] == pid][["거래처_ID","거래처명"]].values
    print(f"  [{pid}] {p_name} ({row['node_type']})")
    for kid_id, kid_name in kids[:5]:
        print(f"       └─ [{int(kid_id)}] {kid_name}")
    if len(kids) > 5:
        print(f"       └─ ... 외 {len(kids)-5}개")

# parent_id가 존재하지 않는 ID를 가리키는 행 확인 (정합성 체크)
valid_ids = set(df_master["거래처_ID"])
broken = df_master[df_master["parent_id"].notna() & ~df_master["parent_id"].isin(valid_ids)]
print(f"\n깨진 parent_id 참조: {len(broken)}개")
if len(broken) > 0:
    print(broken[["거래처_ID","거래처명","parent_id","parent_명"]])

【Parent → Child 샘플】
  [508] 청호나이스 (고객사)
       └─ [855] 미래씨엔에이치
       └─ [856] 에스엠나이스
       └─ [857] 협신산업
  [632] CJ대한통운 (고객사)
       └─ [669] 가미락
       └─ [670] 강원도옥수수총각
       └─ [671] 그림엘푸드
       └─ [672] 남천영농조합법인
       └─ [673] 도당
       └─ ... 외 21개
  [634] 동원홈푸드 (고객사)
       └─ [718] 가산공장
       └─ [719] 금천미트/대전
       └─ [720] 대전센터
       └─ [721] 시화센터
       └─ [722] 장호원
       └─ ... 외 1개
  [659] 한국파렛트풀 (고객사)
       └─ [893] 대상
       └─ [894] 더다냉동물류
       └─ [895] 더유리빙
       └─ [896] 동원로엑스
       └─ [897] 두레냉동
       └─ ... 외 18개
  [594] 한국이노팩 (고객사)
       └─ [890] 이푸드부산
       └─ [891] 이푸드평택
       └─ [892] 현대그린푸드
  [654] 컬리 (고객사)
       └─ [858] 남양주
       └─ [859] 송파

깨진 parent_id 참조: 0개


In [9]:
# ─────────────────────────────────────────────
# 8. Excel 저장 (Master만 — Detail은 단가 테이블과 함께 추후 작업)
# ─────────────────────────────────────────────
OUTPUT_PATH = "SQL_계층구조_거래처DB.xlsx"

COLORS = {
    "header"      : "1B2A3B",
    "고객사_bg"   : "D6EAF8", "고객사_font": "1A5276",
    "납품처_bg"   : "D5F5E3", "납품처_font": "1E8449",
}

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_master.to_excel(writer, sheet_name="Master_거래처", index=False)

    ws = writer.sheets["Master_거래처"]

    # 헤더
    h_fill = PatternFill("solid", start_color=COLORS["header"], end_color=COLORS["header"])
    h_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    for cell in ws[1]:
        cell.fill = h_fill
        cell.font = h_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

    # 행 색상 (node_type 기준)
    nt_col = df_master.columns.get_loc("node_type") + 1
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        nt   = ws.cell(row=row_idx, column=nt_col).value
        bg   = COLORS.get(f"{nt}_bg",   "FFFFFF")
        fc   = COLORS.get(f"{nt}_font", "000000")
        bold = (nt == "고객사")
        fill = PatternFill("solid", start_color=bg, end_color=bg)
        font = Font(color=fc, bold=bold, name="Arial", size=9)
        for cell in row:
            cell.fill = fill
            cell.font = font

    # 청구대상_ID 컬럼 강조 (주황색 헤더)
    bill_col = df_master.columns.get_loc("청구대상_ID") + 1
    ws.cell(row=1, column=bill_col).fill = PatternFill("solid", start_color="E67E22", end_color="E67E22")

    # 열 너비
    for col in ws.columns:
        w = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(w + 4, 45)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

print(f"✅ 저장 완료: {OUTPUT_PATH}")

PermissionError: [Errno 13] Permission denied: 'SQL_계층구조_거래처DB.xlsx'

---
---
## 📦 출하자료 신규 거래처 분석 및 Master 업데이트
> `복사본_출하증_2023_원본_2025_.xlsm` → 출하자료 시트
>
> 기존 V3 Master에 없는 거래처/납품처 조합을 추출해서 append

In [825]:
# ══════════════════════════════════════════════
# 셀 A: 출하자료 로드 + 정제 (pandas)
# ══════════════════════════════════════════════
CHULHA_PATH = "복사본 출하증 2023 원본(2025).xlsm"

df_chulha = pd.read_excel(
    CHULHA_PATH,
    sheet_name="출하자료",
    engine="openpyxl",
    usecols=["거래처", "납품처"],
    dtype=str
)

print(f"출하자료 원본 행 수: {len(df_chulha)}")

# ── 기본 정제 ──────────────────────────────────
df_chulha = df_chulha.fillna("").apply(lambda col: col.str.strip())
df_chulha_og = df_chulha.copy() 

출하자료 원본 행 수: 220807


In [ ]:
df_chulha = df_chulha_og.copy()  

# ── 노이즈 제거 ────────────────────────────────
NOISE = {
    "", "None", "nan", "SAMPLE", "MA", "SET",
    "M500100087/M500100086", "M500100088/M500100089", "M100(A라인)",
    "T543MA-ADF", "T543MA-UNDER", "T539-BASE", "T539-FIN", "T539-T/B",
    "ENV-200BS", "TAC510-20SU/B", "TAC510-20S", "성림BOX(몸통,뚜껑따로)", "BN69-17268A"
    # "1층", "2층", "3층", "지하",
    # "1공장", "2공장", "1센터", "2센터",
    # "1층,중간", "1층 ,중간", "1층사무실 앞", "12동",
}

def is_noise(val: str) -> bool:
    if val in NOISE: return True
    if val.replace("-", "").replace(".", "").isdigit(): return True
    # DW-1호#2, SI-14호 류 (영문+숫자+호) # 여기서 아이템들 싹 걸러냄 호, 호# 만!
    if re.search(r'\d+호(#\d+)?$', val): return True
    # SR-BOT(60배), T595-BOT(60배), 나노박스(64배) 류 (배수 표기)
    if re.search(r'\(\d+배\)', val): return True
    # HD1-TOP#2, KURLY-1#2, SI-8K#2 류 (#숫자로 끝나는 코드)
    if re.search(r'#\d+$', val): return True
    return False

# 거래처가 노이즈인 행 제거
df_chulha = df_chulha[~df_chulha["거래처"].apply(is_noise)].copy()
# 납품처 노이즈는 빈 문자열로
df_chulha["납품처"] = df_chulha["납품처"].apply(lambda x: "" if is_noise(x) else x)

# ── TYPO_MAP / 오타수정만 / strip_inggo ───────
# 괄호는 건드리지 않음
def fix_typo_chulha(name: str) -> str:
    """출하자료 전용 — 오타 수정만, 괄호 변환/제거 없음"""
    name = name.strip()
    if "/" in name:
        parts = [TYPO_MAP.get(p.strip(), p.strip()) for p in name.split("/", 1)]
        return "/".join(parts)
    return TYPO_MAP.get(name, name)

# 사실 fix_typo_chulha까지는 괜찮은데 strip_inggo가 / 아니라서 의미가 없음...
df_chulha["거래처"] = df_chulha["거래처"].apply(fix_typo_chulha).apply(strip_inggo)
df_chulha["납품처"] = df_chulha["납품처"].apply(fix_typo_chulha).apply(strip_inggo)

# ── 거래처 슬래시 정리 ─────────────────────────
def expand_slash(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        client, delivery = row["거래처"], row["납품처"]
        if "/" in client:
            parent = client.split("/")[0].strip()
            child  = client.split("/", 1)[1].strip()
            # 슬래시 뒤를 납품처로 추가
            rows.append({"거래처": parent, "납품처": child})
            # 기존 납품처도 살리기 (다를 경우)
            if delivery and delivery != child:
                rows.append({"거래처": parent, "납품처": delivery})
        else:
            rows.append({"거래처": client, "납품처": delivery})
    return pd.DataFrame(rows)

df_chulha = expand_slash(df_chulha).drop_duplicates().reset_index(drop=True)

# ── 납품처 슬래시 → 행 분리 ───────────────────
def expand_delivery_slash(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        delivery = row["납품처"]
        if "/" in delivery:
            for d in delivery.split("/"):
                d = d.strip()
                if d:
                    rows.append({"거래처": row["거래처"], "납품처": d})
        else:
            rows.append({"거래처": row["거래처"], "납품처": delivery})
    return pd.DataFrame(rows)

df_chulha = expand_delivery_slash(df_chulha).drop_duplicates().reset_index(drop=True)

# ── 납품처 끝자락에있는 (주),(유) 임시 저장 ─────────────
df_chulha["_suffix"] = df_chulha["납품처"].apply(
    lambda x: re.search(r'\((주|유)\)$', x).group() if re.search(r'\((주|유)\)$', x) else ""
)
df_chulha["납품처"] = df_chulha["납품처"].apply(
    lambda x: re.sub(r'\((주|유)\)$', '', x).strip()
)

# ── 납품처 괄호 → 행 분리 ───────────────────── 끝에있는 (주)(유)를 뺴야 ()안에있는걸 정재할수있음..
def expand_delivery_parenthesis(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        match = re.match(r'^(?!\()(.+?)\((.+?)\)$', row["납품처"])
        if match:
            rows.append({"거래처": row["거래처"], "납품처": match.group(1).strip(), "_suffix": row["_suffix"]})
            rows.append({"거래처": row["거래처"], "납품처": match.group(2).strip(), "_suffix": ""})
        else:
            rows.append({"거래처": row["거래처"], "납품처": row["납품처"], "_suffix": row["_suffix"]})
    return pd.DataFrame(rows)

df_chulha = expand_delivery_parenthesis(df_chulha)

# ── suffix 복원 ────────────────────────────────
df_chulha["납품처"] = df_chulha.apply(
    lambda row: row["납품처"] + row["_suffix"] if row["_suffix"] else row["납품처"], axis=1
)
df_chulha = df_chulha.drop(columns=["_suffix"]).drop_duplicates().reset_index(drop=True)


# 거래처 괄호 → 제거 (맨 앞 제외)
# 본수원갈비(본점) → 본수원갈비
# 연우로지스(선우) → 연우로지스
# (주)진보 → 그대로
df_chulha["거래처"] = df_chulha["거래처"].apply(
    lambda x: x if re.match(r'^\(', x) else re.sub(r'\(.*?\)', '', x).strip()
)
df_chulha = df_chulha.drop_duplicates().reset_index(drop=True)
print(f"정리 후: {len(df_chulha)}")


정리 후: 3519


In [873]:
df_chulha[df_chulha['거래처'] == '용인공장']

,거래처,납품처
2240,용인공장,용인공장
2253,용인공장,태성산업
2256,용인공장,천우에프디
2257,용인공장,한일전기
2263,용인공장,유준수지
2266,용인공장,정진에프피씨
2269,용인공장,청원오가닉
2271,용인공장,바다원
2285,용인공장,영광포장
2298,용인공장,우성AFC


In [874]:
df_chulha_before_promote_inggo = df_chulha.copy()
df_chulha_before_promote_inggo

,거래처,납품처
0,둔포공장,둔포공장
1,중부스티로폼,㈜이삭
2,중부스티로폼,중부스티로폼
3,LG전자,C07
4,LG전자,C02
...,...,...
3514,중부스티로폼,이삼우블루베리농장
3515,아이센스,안산
3516,아이센스,판토스
3517,한국이노팩,천안블루베리


In [ ]:
# 입고처가 거래처인 경우 → 납품처를 거래처로 올려버리기
def promote_inggo(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        if row["거래처"] in INGGO_SET:
            # 납품처가 있으면 그걸 거래처로 승격
            if row["납품처"]:
                rows.append({"거래처": row["납품처"], "납품처": ""})
        else:
            rows.append({"거래처": row["거래처"], "납품처": row["납품처"]})
    return pd.DataFrame(rows)

df_chulha = promote_inggo(df_chulha)

# 거래처 == 납품처 → 납품처 빈 문자열로
df_chulha.loc[df_chulha["거래처"] == df_chulha["납품처"], "납품처"] = ""
df_chulha = df_chulha.drop_duplicates().reset_index(drop=True)

In [894]:
# ══════════════════════════════════════════════
# 셀 B: 기존 Master와 비교 → 신규 거래처/납품처 확인
# ══════════════════════════════════════════════
existing = set(df_master["거래처명"].unique())

# 신규 거래처 (Master에 없는 것)
df_new_clients = df_chulha[
    ~df_chulha["거래처"].isin(existing)
].drop_duplicates(subset=["거래처"]).reset_index(drop=True)

# 기존 거래처인데 납품처가 신규인 것
df_new_delivery = df_chulha[
    df_chulha["거래처"].isin(existing) &
    (df_chulha["납품처"] != "") &
    ~df_chulha["납품처"].isin(existing)
].drop_duplicates().reset_index(drop=True)

print(f"기존 Master: {len(existing)}개")
print(f"신규 거래처: {len(df_new_clients)}개")
print(f"기존 거래처의 신규 납품처 조합: {len(df_new_delivery)}개")
print()
print("【신규 거래처】")
df_new_clients


기존 Master: 889개
신규 거래처: 232개
기존 거래처의 신규 납품처 조합: 2147개

【신규 거래처】


,거래처,납품처
0,헬로네이쳐,
1,더파머스,
2,아트박스,
3,LGMC,
4,미래원,
...,...,...
227,와이제이글로벌,일양로지스
228,혜천식품,
229,엘에프푸드,
230,컬리건코리아,


In [942]:
# ══════════════════════════════════════════════
# 셀 D: Master에 append + 저장
# ══════════════════════════════════════════════
# ※ 셀 B/C 결과 확인 후 실행
#   제외할 거래처는 EXCLUDE에 추가
EXCLUDE = set()

# 신규 거래처 → 고객사로 추가
df_add_clients = df_new_clients[
    ~df_new_clients["거래처"].isin(EXCLUDE)
].copy()
df_add_clients = df_add_clients.rename(columns={"거래처": "거래처명"})
df_add_clients["parent_명"] = None
df_add_clients["node_type"] = "고객사"
for col in ["대표자","사업자번호","주소","업태","종목","납품주소",
            "마감담당자_성명","마감담당자_이메일","마감담당자_연락처",
            "실무담당자_성명","실무담당자_이메일","실무담당자_연락처",
            "운임","결제조건_일"]:
    df_add_clients[col] = None
df_add_clients = df_add_clients.drop(columns=["납품처"], errors="ignore")

# 신규 납품처 → 납품처로 추가
df_add_delivery = df_new_delivery[
    ~df_new_delivery["납품처"].isin(EXCLUDE)
].copy()
df_add_delivery = df_add_delivery.rename(columns={"납품처": "거래처명", "거래처": "parent_명"})
df_add_delivery["node_type"] = "납품처"
for col in ["대표자","사업자번호","주소","업태","종목","납품주소",
            "마감담당자_성명","마감담당자_이메일","마감담당자_연락처",
            "실무담당자_성명","실무담당자_이메일","실무담당자_연락처",
            "운임","결제조건_일"]:
    df_add_delivery[col] = None

# Master에 append
df_master_updated = pd.concat(
    [df_master, df_add_clients, df_add_delivery],
    ignore_index=True
)

# ID 재부여
df_master_updated["거래처_ID"] = range(1, len(df_master_updated) + 1)
name_to_id_upd = df_master_updated.set_index("거래처명")["거래처_ID"].to_dict()
df_master_updated["parent_id"]   = df_master_updated["parent_명"].map(name_to_id_upd)
df_master_updated["청구대상_ID"] = df_master_updated.apply(
    lambda row: row["parent_id"]
        if (row["node_type"] == "납품처" and pd.notna(row["parent_id"]))
        else row["거래처_ID"],
    axis=1
).astype("Int64")

print(f"기존 Master : {len(df_master)}개")
print(f"신규 고객사  : {len(df_add_clients)}개")
print(f"신규 납품처  : {len(df_add_delivery)}개")
print(f"업데이트 후  : {len(df_master_updated)}개")


기존 Master : 921개
신규 고객사  : 232개
신규 납품처  : 2147개
업데이트 후  : 3300개


In [943]:

# ── Excel 저장 ──────────────────────────────────
OUTPUT_UPDATED = "SQL_계층구조_거래처DB_updated.xlsx"
with pd.ExcelWriter(OUTPUT_UPDATED, engine="openpyxl") as writer:
    df_master_updated.to_excel(writer, sheet_name="Master_거래처", index=False)

    ws = writer.sheets["Master_거래처"]
    COLORS = {
        "header"    : "1B2A3B",
        "고객사_bg" : "D6EAF8", "고객사_font": "1A5276",
        "납품처_bg" : "D5F5E3", "납품처_font": "1E8449",
        "신규_bg"   : "FEF9E7",
    }
    h_fill = PatternFill("solid", start_color=COLORS["header"], end_color=COLORS["header"])
    h_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    for cell in ws[1]:
        cell.fill = h_fill
        cell.font = h_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

    nt_col        = df_master_updated.columns.get_loc("node_type") + 1
    new_start_row = len(df_master) + 2

    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        nt = ws.cell(row=row_idx, column=nt_col).value
        if row_idx >= new_start_row:
            bg, fc, bold = COLORS["신규_bg"], "7D6608", False
        else:
            bg   = COLORS.get(f"{nt}_bg",   "FFFFFF")
            fc   = COLORS.get(f"{nt}_font", "000000")
            bold = (nt == "고객사")
        fill = PatternFill("solid", start_color=bg, end_color=bg)
        font = Font(color=fc, bold=bold, name="Arial", size=9)
        for cell in row:
            cell.fill = fill
            cell.font = font

    bill_col = df_master_updated.columns.get_loc("청구대상_ID") + 1
    ws.cell(row=1, column=bill_col).fill = PatternFill("solid", start_color="E67E22", end_color="E67E22")

    for col in ws.columns:
        w = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(w + 4, 45)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

print(f"\n✅ 저장 완료: {OUTPUT_UPDATED}")
print("  🟡 노란색 행 = 출하자료에서 신규 추가된 거래처")


✅ 저장 완료: SQL_계층구조_거래처DB_updated.xlsx
  🟡 노란색 행 = 출하자료에서 신규 추가된 거래처
